# MAI-AHLT

### Josep de Cid & Víctor Giménez

In [1]:
import os

os.chdir(os.path.join('..', '..'))

Run the following command in the terminal to launch the CoreNLP server:

```java -mx4g -cp "*" edu.stanford.nlp.pipeline.StanfordCoreNLPServer```

To allow full reproducibility we remark that we are using the 4.0.0 version released on 2020-04-19.

In [2]:
from nltk.parse.corenlp import CoreNLPDependencyParser

DEPENDENCY_PARSER = CoreNLPDependencyParser(url='http://localhost:9000')

We define some global constants such as the data directory and evaluation JAR paths, and the name of the output file.

In [3]:
TRAIN_DIR = 'project/data/Train'
DEVEL_DIR = 'project/data/Devel'
TEST_DIR = 'project/data/Test-DDI'

OUTPUT_FILE = 'task9.2_VJ_42.txt'

EVAL_JAR_DIR = os.path.join('project', 'eval', 'evaluateDDI.jar')

There are some special symbols (parentheses, brackets and keys) that are mapped into special tokens such as `-LRB-` for `(`. We define an inverse mapping to find which are these symbols in our data.

In [4]:
MAP_SPECIAL_SYMBOLS = {
    '-LRB-': '(',
    '-RRB-': ')',
    '-LSB-': '[',
    '-RSB-': ']',
    '-LCB-': '{',
    '-RCB-': '}',
}

## Task 3 - Rule Based DDI

The method `parse_folder` reads the XML files inside a folder, parsing those to extract the sentences and the corresponding `entity`(es) and `pair`(s) associated. Then we call auxiliar methods to extrect and detect the DDI.

It returns either a string or a tuple of strings to abstract this method and use it for both tasks 3 and 4 as we will describe later.

In [5]:
from glob import glob
from typing import Union, List, Tuple

from bs4 import BeautifulSoup as Bs
from tqdm.notebook import tqdm


def parse_folder(input_dir: str, task: int) -> Union[str, Tuple[str, str]]:
    """
    Parses a folder with XML files containing sentences with their corresponding entities and pairs.
    
    Parameters
    ----------
    input_dir : str
        Path to the data folder.
    task : int
        AHLT lab session task to perform (3 or 4)
        
    Returns
    -------
    features : Union[str, Tuple[str, str]]
        Features in string format for each o the sentences in the folder. For the case of task 4,
        two strings are returned, one for the binary and the other for the multiclass classification.
    """
    assert task == 3 or task == 4
    output_data = '' if task == 3 else ['', '']
    
    for file in tqdm(glob(os.path.join(input_dir, '*')), ncols=800, desc='Detecting DDI'):
        with open(file, mode='r') as f:
            content = f.read().strip()

            # Parse XML file, obtaining a DOM tree
            tree = Bs(content, 'lxml')
            
            for sentence in tree.find_all('sentence'):
                # Skip empty sentences
                if sentence['text'] == '':
                    continue
                
                # Load sentence entities into a dictionary
                entities = {}
                sentence_entities = sentence.find_all('entity')
                for entity in sentence_entities:
                    if ';' in entity['charoffset']:
                        parts = entity['charoffset'].split(';')
                        offs = [parts[0].split('-')[0], parts[-1].split('-')[1]]
                    else:
                        offs = entity['charoffset'].split('-')
                    entities[entity['id']] = int(offs[0]), int(offs[1])

                # Tokenize, tag, and parse sentence
                analysis = analyze(sentence['text'])
                
                # For each `pair` in the sentence, decide whether it is DDI and its type
                for pair in sentence.find_all('pair'):
                    
                    # This part is extracted to an external method to be reused by both tasks.
                    pair_result = task_features(analysis, entities, pair, sentence['id'])
                    
                    if task == 3:
                        output_data += pair_result
                    else:
                        output_data[0] += pair_result[0]
                        output_data[1] += pair_result[1]
    
    return output_data if task == 3 else tuple(output_data)

Now, let's define the method to analyze the tokens and extract them, along with the tags and dependency tree using the CoreNLP server. This method will be used by both tasks.

In [6]:
import re
from typing import Dict


def analyze(sentence: str) -> Dict[int, Dict[str, Union[str, int]]]:
    """
    Extracts the tokens, tags, and dependency tree from a sentence.
    
    Parameters
    ----------
    sentence : str
        Raw sentence as a string.
        
    Returns
    -------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Analysis in a dictionary structure where the keys are just the indices of the sentence,
        and the values are another dictionary containing the token information. These elements are:
        `head`, `word`, `rel`, `tag`, and `lemma`.
    """
    # Replace strange HTML and breakline symbols interpreted as multiple characters
    sentence = re.sub(r'&#..;', '|', sentence)
    sentence = sentence.replace('\n', '|')
    sentence = sentence.replace('\r', '|')
    
    tree, = DEPENDENCY_PARSER.raw_parse(sentence)
    nodes = tree.nodes
    
    result = {}
    for key, content in nodes.items():
        result[key] = {k: content[k] for k in ['head', 'word', 'rel', 'tag', 'lemma']}

    i = 0
    for key in range(len(result)):
        word = result[key]['word']
        if word is not None:
            if word in MAP_SPECIAL_SYMBOLS.keys():
                word = MAP_SPECIAL_SYMBOLS[word]

            word_pos = sentence.find(word, i)

            result[key]['start'] = word_pos
            result[key]['end'] = word_pos + len(word) - 1

            i = word_pos + len(word)

    return result

The following method defines the data to be obtained within this task, using an auxiliar function defined after it named `check_interaction`, which at the same time, uses the auxiliar methods `entity_to_indices`, `extract_head`, `extract_parents` and `least_common_parent`.

With all these methods we are able to identify if there is a drug-drug interaction between the given entities using heuristic hand-crafted rules.

In [7]:
def task_features(analysis: Dict[int, Dict[str, Union[str, int]]], entities: Dict[str, Tuple[int, int]],
                  pair: Dict[str, str], s_id: str) -> str:
    """
    Extract features and outputs them along with the label.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    entities : Dict[str, Tuple[int, int]]
        List of entities with the associated offsets in the sentence.
    pair : Dict[str, str]
        Potential DDI pair in the corresponding sentence.
    s_id : str
        Sentence ID to generate the output format.
    
    Returns
    -------
    classifier_out : str
        Output features in string format for the DDI multiclass classification.
    """
    (is_ddi, ddi_type) = check_interaction(analysis, entities, pair['e1'], pair['e2'])
    return '|'.join([s_id, pair['e1'], pair['e2'], is_ddi, ddi_type]) + '\n'


def check_interaction(analysis: Dict[int, Dict[str, Union[str, int]]], entities: Dict[str, Tuple[int, int]],
                      e1: str, e2: str) -> Tuple[str, str]:
    """
    Checks the interaction between two entities using heuristic hand-crafted rules.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    entities : Dict[str, Tuple[int, int]]
        List of entities with the associated offsets in the sentence.
    e1 : str
        First entity to analyze.
    e2 : str
        Second entity to analyze.
        
    Returns
    -------
    interaction : Tuple[str, str]
        Interaction result in a tuple, where the first element is binary '1' or '0', and the second is
        the corresponding class (if the first one is '1').
    """
    try:
        range_e1 = entity_to_indices(analysis, entities[e1])
        range_e2 = entity_to_indices(analysis, entities[e2])
    except AssertionError:
        # Error in some multi-part entities (a-b;c-d)
        return '0', 'null'

    head_e1 = extract_head(analysis, range_e1)
    head_e2 = extract_head(analysis, range_e2)

    lcp = least_common_parent(analysis, head_e1, head_e2)

    # We define some keywords that may have more importance for each of the different classes.
    keywords_by_type = {
        'effect': ['administer', 'potentiate', 'prevent'],
        'mechanism': ['reduce', 'increase', 'decrease'],
        'advise': ['should', 'recommend', 'avoid'],
        'int': ['interact', 'interaction']
    }

    # If any of the entities of the sentence matches any of the keywords (in order) we return these class.
    for y_type, keywords in keywords_by_type.items():
        if any(map(lambda key: analysis[key]['lemma'] in keywords, extract_parents(analysis, lcp))):
            return '1', y_type

    # Otherwise, we return '0' with a 'null' tag on the second element.
    return '0', 'null'

def entity_to_indices(analysis: Dict[int, Dict[str, Union[str, int]]],
                      entity_offset: Tuple[int, int]) -> Tuple[int, int]:
    """
    Extract the indices from the entity.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    entity_offset : Tuple[int, int]
        Offsets of the entity from the parsed XML file.
    
    Return
    ------
    indices : Tuple[int, int]
        Indices of the entity.
    """
    start, end = None, None
    for key, value in analysis.items():
        if key == 0:
            continue
        if value['start'] <= entity_offset[0] <= value['end']:
            start = key
        if value['start'] <= entity_offset[1] <= value['end']:
            end = key
        if start is not None and end is not None:
            break
    
    assert start is not None and end is not None, f'{analysis} ||| {entity_offset}'
    return start, end


def extract_head(analysis: Dict[int, Dict[str, Union[str, int]]], indices: Tuple[int, int]) -> int:
    """
    Extract the head corresponding to the given indices.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    indices : Tuple[int, int]
        Offset of the head to locate.
    
    Returns
    -------
    head : int
        Corresponding head.
    """
    for i in range(indices[0], indices[1] + 1):
        head = analysis[i]['head']
        if head < indices[0] or head > indices[1]:
            return head


def extract_parents(analysis: Dict[int, Dict[str, Union[str, int]]], head: int) -> List[int]:
    """
    Extract the parents of the given head in ascendant order.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    head : int
        Head of the pair entity to extract the parents from.
        
    Returns
    -------
    parents : List[int]
        List of parents in ascendant order.
    """
    parent = head
    parents = []
    while parent is not None:
        parents.append(parent)
        parent = analysis[parent]['head']
    return parents


def least_common_parent(analysis: Dict[int, Dict[str, Union[str, int]]], head1: int, head2: int) -> int:
    """
    Extract the leaset_common_parent of two heads.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    head1 : int
        Head of the first pair entity to use.
    head1 : int
        Head of the second pair entity to use.
        
    Returns
    -------
    parent : int
        ID of the least common parent.
    """
    parents1 = extract_parents(analysis, head1)
    parents2 = extract_parents(analysis, head2)

    for parent in parents1:
        if parent in parents2:
            return parent

For the rules, after simply taking a simple look at the Training set, we figured out that there are a series of keywords for each of the classes. After the usage of several functions, we extract all of the 'common parents' in the parsed sentence tree, and if one of the keywords is in that set, we classify them as the keyword's class. We order the search in that set such that we achieve maximum accuracy in train and devel. Since we hit a pretty high accuracy (for the baseline) we did not try nor discard many more rules.

Now, we just need to define the method to evaluate the results with the given JAR file and the main method to be called with the desired folders and files as the input parameters.

In [8]:
import subprocess


def evaluate(input_dir: str, output_dir: str) -> None:
    """
    Calls the evaluator provided in the Java JAR DDI file.
    
    Parameters
    ----------
    input_dir : str
        Path to the data folder to evaluate.
    output_dir : str
        Path to the output file with the predictions.
    """
    result = subprocess.run(args=['java', '-jar', EVAL_JAR_DIR, input_dir, output_dir],
                            stdout=subprocess.PIPE)
    print(result.stdout.decode('utf8'))
    

def interaction_classifier(input_dir: str, output_file: str) -> None:
    """
    Main class to detect Drug-Drug interactions for Task 3 using heuristic hand-crafted rules.
    
    Parameters
    ----------
    input_dir : str
        Path to the data folder.
    output_file : str
        Path to the output file used by the DDI JAR.
    """
    output_data = parse_folder(input_dir, 3)
    
    with open(output_file, mode='w') as f:
        f.write(output_data)

    evaluate(input_dir, output_file)

In [9]:
interaction_classifier(DEVEL_DIR, OUTPUT_FILE)


Gold project/data/Devel
Submission  task9.2_VJ_42.txt

project/data/Devel
 Gold DDI standard saved in goldDDI.txt total pairs: 2703
SCORES FOR THE GROUP: VJ RUN=42
Gold Dataset: /Devel

Partial Evaluation: only detection of DDI (regadless to the type)
tp	fp	fn	total	prec	recall	F1
157	357	327	484	0.3054	0.3244	0.3146


Detection and classification of DDI
tp	fp	fn	total	prec	recall	F1
86	428	398	484	0.1673	0.1777	0.1723


________________________________________________________________________

SCORES FOR DDI TYPE
Scores for ddi with type mechanism
tp	fp	fn	total	prec	recall	F1
55	169	146	201	0.2455	0.2736	0.2588


Scores for ddi with type effect
tp	fp	fn	total	prec	recall	F1
14	168	148	162	0.0769	0.0864	0.0814


Scores for ddi with type advise
tp	fp	fn	total	prec	recall	F1
15	16	104	119	0.4839	0.1261	0.2


Scores for ddi with type int
tp	fp	fn	total	prec	recall	F1
2	75	0	2	0.026	1	0.0506


MACRO-AVERAGE MEASURES FOR DDI CLASSIFICATION:
	P	R	F1
	0.2081	0.3715	0.2668
___________________

In [10]:
interaction_classifier(TRAIN_DIR, OUTPUT_FILE)


Gold project/data/Train
Submission  task9.2_VJ_42.txt

project/data/Train
 Gold DDI standard saved in goldDDI.txt total pairs: 25089
SCORES FOR THE GROUP: VJ RUN=42
Gold Dataset: /Train

Partial Evaluation: only detection of DDI (regadless to the type)
tp	fp	fn	total	prec	recall	F1
1208	6553	2329	3537	0.1557	0.3415	0.2138


Detection and classification of DDI
tp	fp	fn	total	prec	recall	F1
717	7044	2820	3537	0.0924	0.2027	0.1269


________________________________________________________________________

SCORES FOR DDI TYPE
Scores for ddi with type mechanism
tp	fp	fn	total	prec	recall	F1
321	3641	797	1118	0.081	0.2871	0.1264


Scores for ddi with type effect
tp	fp	fn	total	prec	recall	F1
164	1028	1361	1525	0.1376	0.1075	0.1207


Scores for ddi with type advise
tp	fp	fn	total	prec	recall	F1
96	216	611	707	0.3077	0.1358	0.1884


Scores for ddi with type int
tp	fp	fn	total	prec	recall	F1
136	2159	50	186	0.0593	0.7312	0.1096


MACRO-AVERAGE MEASURES FOR DDI CLASSIFICATION:
	P	R	F1
	0.1464	0

In [11]:
interaction_classifier(TEST_DIR, OUTPUT_FILE)


Gold project/data/Test-DDI
Submission  task9.2_VJ_42.txt

project/data/Test-DDI
 Gold DDI standard saved in goldDDI.txt total pairs: 5716
SCORES FOR THE GROUP: VJ RUN=42
Gold Dataset: /Test-DDI

Partial Evaluation: only detection of DDI (regadless to the type)
tp	fp	fn	total	prec	recall	F1
356	1261	623	979	0.2202	0.3636	0.2743


Detection and classification of DDI
tp	fp	fn	total	prec	recall	F1
202	1415	777	979	0.1249	0.2063	0.1556


________________________________________________________________________

SCORES FOR DDI TYPE
Scores for ddi with type mechanism
tp	fp	fn	total	prec	recall	F1
84	548	218	302	0.1329	0.2781	0.1799


Scores for ddi with type effect
tp	fp	fn	total	prec	recall	F1
23	183	337	360	0.1117	0.0639	0.0813


Scores for ddi with type advise
tp	fp	fn	total	prec	recall	F1
42	115	179	221	0.2675	0.19	0.2222


Scores for ddi with type int
tp	fp	fn	total	prec	recall	F1
53	569	43	96	0.0852	0.5521	0.1476


MACRO-AVERAGE MEASURES FOR DDI CLASSIFICATION:
	P	R	F1
	0.1493	0.271	0.1

## Task 4 - Machine Learning DDI

For this task we will extract features from the data instead of using heuristics, using a classification Machine Learning algorithm to solve the problem. The following method extracts these features, described via line comments before their extraction.

We took into consideration several things: first, the lemmas (depending on their position in the sentence). After seeing the importance of the parents in the baseline, we also added features regarding the entity's direct parents (and the relationship to them), and all of the common parents of the entities. Furthermore, we set a special feature for when one entity is parent of the other. We add one rule also for explicifying the verb lemmas of the sentence. Finally, since we struggled with the 'interaction' class, we added a feature for explicifying when there is a word starting by 'interact' in the sentence, which improved our result a bit.
We didn't discard many features. All of the things we added gave a small boost, with the exception of information of the parent's parent, information about affixes (which we believed could help reduce sparsity hopefully), and much more hand-crafted rules (such as for when one entity is the direct object of the verb).

In [12]:
def extract_features(analysis, entities: Dict[str, Tuple[int, int]], e1: str, e2: str) -> List[str]:
    """
    Extracts features from a pair of entities.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    entities : Dict[str, Tuple[int, int]]
        List of entities with the associated offsets in the sentence.
    e1 : str
        First entity.
    e2 : str
        econd entity.
    
    Returns
    -------
    features : List[str]
        Vector of binary features in string format
    """
    # We initialize the feature vector empty
    feat_vector = []

    try:
        range_e1 = entity_to_indices(analysis, entities[e1])
        range_e2 = entity_to_indices(analysis, entities[e2])
    except AssertionError:
        # Error in some multi-part entities (a-b;c-d)
        return feat_vector
    
    # There are very few `int` type interactions, that may downgrade our score if missclassified
    # To enforce a better performance in this class we enforce a feature that contains the keyword
    # interact as a lemma in any part of the sentence. 
    for token in analysis.values():
        if token['lemma'] is not None and 'interact' in token['lemma']:
            feat_vector.append('interact')
    
    # Lemma before entity 1
    for i in range(1, range_e1[0]):
        lemma = analysis[i]['lemma']
        if lemma.isalnum() and not lemma.isnumeric():
            feat_vector.append(f'lb1={lemma}')
        
    # Lemma between both entities
    for i in range(range_e1[1] + 1, range_e2[0]):
        lemma = analysis[i]['lemma']
        if lemma.isalnum() and not lemma.isnumeric():
            feat_vector.append(f'lib={lemma}')
     
    # Lemma after entity 2
    for i in range(range_e2[1] + 1, len(analysis)):
        lemma = analysis[i]['lemma']
        if lemma.isalnum() and not lemma.isnumeric():
            feat_vector.append(f'la2={lemma}')

    head_e1 = extract_head(analysis, range_e1)
    parent_head_e1 = analysis[head_e1]['head']
    head_e2 = extract_head(analysis, range_e2)
    parent_head_e2 = analysis[head_e2]['head']

    # Relation with the parent of the entity and its lemma for entity 1
    if parent_head_e1 is not None:
        feat_vector += [
            f'entity1_rel={analysis[head_e1]["rel"]}',
            f'entity1_parent_lemma={analysis[parent_head_e1]["lemma"]}'
        ]

    # Relation with the parent of the entity and its lemma for entity 2
    if parent_head_e2 is not None:
        feat_vector += [
            f'entity2_rel={analysis[head_e2]["rel"]}',
            f'entity2_parent_lemma={analysis[parent_head_e2]["lemma"]}',
        ]

    lcp = least_common_parent(analysis, head_e1, head_e2)
    cp = extract_parents(analysis, lcp)
    
    # Lemmas of common parents for both entities
    feat_vector += [f'parent={analysis[i]["lemma"]}' for i in cp]
    
    # One entity is parent of the other (any of both directions)
    if head_e1 in cp or head_e2 in cp:
        feat_vector += ['one_parent_of_other']

    # The least common parent has direct parenthood with the corresponding head
    if lcp == head_e1:
        feat_vector += [f'direct_parenthood={analysis[head_e2]["rel"]}']
    if lcp == head_e2:
        feat_vector += [f'direct_parenthood={analysis[head_e1]["rel"]}']

    # Lemmas of verbs in common parents
    for p in cp:
        if analysis[p]['tag'] == 'VB':
            feat_vector += [f'verb={analysis[p]["lemma"]}']

    return feat_vector

For this task, the goal is to generate a features file that is later used to feed a model that predicts the interactions between pairs of drugs. For the training data (if available) we join the ground class with the extracted features using the previous method. For the test, we can ignore the true label as the JAR file already evaluates the correctness of our program.

Instead of training and performing a single classification between no-interaction and the other classes, we split this process in a hierarchical process where first of all we determine if there is an interaction or not, to later on, if it is an interaction, decide of which type. We extract and use the same features for both models, but one is trained as a binary classification problem while the other one is a multi-class classification problem.

In `binary_out` variable we just put the features along with 'DDI' if there is an interaction or 'null' if note. In an affirmative case, we also output data in `classifier_out` variable, defining the same features but specifiying which kind of interaction on the class.

In [13]:
def task_features(analysis: Dict[int, Dict[str, Union[str, int]]], entities: Dict[str, Tuple[int, int]],
                  pair: Dict[str, str], s_id: str) -> Tuple[str, str]:
    """
    Extract features and outputs them along with the label.
    
    Parameters
    ----------
    analysis : Dict[int, Dict[str, Union[str, int]]]
        Tokens, tags, and dependency tree from the corresponding sentence.
    entities : Dict[str, Tuple[int, int]]
        List of entities with the associated offsets in the sentence.
    pair : Dict[str, str]
        Potential DDI pair in the corresponding sentence.
    s_id : str
        Sentence ID to generate the output format.
    
    Returns
    -------
    binary_out, classifier_out : Tuple[str, str]
        Output features in string format for the binary classification and DDI multiclass classification.
    """
    try:
        ground = pair['type'] if pair['ddi'] == 'true' else 'null'
    except KeyError:
        ground = 'null'
    
    features = extract_features(analysis, entities, pair['e1'], pair['e2'])
    
    # If it is a DDI, output 'true' in the binary file and the corresponding class in the multiclass file
    # Otherwise, output 'false' in the binary file and just skip any output in the other
    if ground != 'null':
        binary_out = '\t'.join([s_id, pair['e1'], pair['e2'], 'DDI', *features]) + '\n'
        classifier_out = '\t'.join([s_id, pair['e1'], pair['e2'], ground, *features]) + '\n'
    else:
        binary_out = '\t'.join([s_id, pair['e1'], pair['e2'], 'null', *features]) + '\n'
        classifier_out = ''
    
    return binary_out, classifier_out

As the learner, we use MegaM to train the models as seen in class. We reuse the class code for the model from the AHLT task [MEM practical example](https://www.cs.upc.edu/~padro/ahlt/lectures/MEM-example.tgz) and get a label for each vector in the dataset using a Maximum Entropy classifier.

We started using MegaM from the beginning, since it is the fastest to interface with binary, string-like binary features. Other classifiers in sklearn would require a much more massive data structure, such as a very sparse FxN table, for every possible feature, and we would've had to tune extra things to make it work. We devoted that time to further explore features. Given the achieved results, as well as the simplicity of the code, we still believe MegaM was the way to go.

In [14]:
from project.models.MEmodel import MEmodel


def model_evaluate():
    """
    Evaluate models with the evaluation features.
    
    Returns
    -------
    classification : str
        Output string formated to feed the JAR DDI program with the corresponding predicted tags.
    """
    bin_model = MEmodel('model_bin.dat')
    class_model = MEmodel('model_class.dat')
    
    output_data = ''
    with open('test_features.dat', mode='r') as f:
        for line in f.readlines():
            data = line.strip().split('\t')
            
            meta = data[:3]
            features = data[4:]

            # Binary classification between DDI or not
            is_ddi = get_best_tag(bin_model, features) == 'DDI'
            
            # Multi-class classification if it is DDI, else just NULL
            tag = get_best_tag(class_model, features) if is_ddi else 'null'
            
            output_data += '|'.join([*meta, '0' if tag == 'null' else '1', tag]) + '\n'
    
    return output_data


def get_best_tag(model: MEmodel, features: List[str]):
    """
    Get the tag with the highest probability.
    
    Parameters
    ----------
    model : MEmodel
        Trained MegaM model.
    features : List[str]
        List of binary string features to evaluate.
    """
    dist = model.prob_dist_z(features)
    best = None
    mx = 0
    for c in dist:
        if dist[c] > mx:
            mx = dist[c]
            best = c
    return best

Now, let's just redefine the main method for this new task. The MegaM training with the binary classification features should be binary but, as it outputs it in a different format, to reuse the code of the model class, we can also train it as multiclass with two classes.

In [15]:
def interaction_classifier(input_dir_train: str, input_dir_eval: str, output_file: str) -> None:
    """
    Main class to detect Drug-Drug interactions for Task 4 using ML.
    
    Parameters
    ----------
    input_dir_train : str
        Path to the training data folder.
    input_dir_eval : str
        Path to the evaluation data folder.
    output_file : str
        Path to the output file used by the DDI JAR.
    """
    # Extract and save train features
    train_bin_features, train_class_features = parse_folder(input_dir_train, task=4)
    
    with open('train_bin_features.dat', mode='w') as f:
        f.write(train_bin_features)
        
    with open('train_class_features.dat', mode='w') as f:
        f.write(train_class_features)
    
    # Remove the first 3 extra fields unnecessary for the megam algorithm
    os.system('cat train_bin_features.dat | cut -f4- > megam_bin.dat')
    os.system('cat train_class_features.dat | cut -f4- > megam_class.dat')

    # Run MegaM to extract model
    os.system('project/models/megam_osx.opt -quiet -nc -repeat 10 multiclass megam_bin.dat > model_bin.dat')
    os.system('project/models/megam_osx.opt -quiet -nc -repeat 10 multiclass megam_class.dat > model_class.dat')
        
    # Extract and save test features
    _, test_features = parse_folder(input_dir_eval, task=4)
    with open('test_features.dat', mode='w') as f:
        f.write(test_features)
    
    # Evaluate, make predictions and store the results
    output_data = model_evaluate()
    with open(output_file, mode='w') as f:
        f.write(output_data)

    # Evaluate the performance using the DDI JAR
    evaluate(input_dir_eval, output_file)

### Results

In [16]:
interaction_classifier(TRAIN_DIR, DEVEL_DIR, OUTPUT_FILE)


Gold project/data/Devel
Submission  task9.2_VJ_42.txt

project/data/Devel
 Gold DDI standard saved in goldDDI.txt total pairs: 2703
SCORES FOR THE GROUP: VJ RUN=42
Gold Dataset: /Devel

Partial Evaluation: only detection of DDI (regadless to the type)
tp	fp	fn	total	prec	recall	F1
415	0	69	484	1	0.8574	0.9232


Detection and classification of DDI
tp	fp	fn	total	prec	recall	F1
380	35	104	484	0.9157	0.7851	0.8454


________________________________________________________________________

SCORES FOR DDI TYPE
Scores for ddi with type mechanism
tp	fp	fn	total	prec	recall	F1
149	15	52	201	0.9085	0.7413	0.8164


Scores for ddi with type effect
tp	fp	fn	total	prec	recall	F1
134	10	28	162	0.9306	0.8272	0.8758


Scores for ddi with type advise
tp	fp	fn	total	prec	recall	F1
95	9	24	119	0.9135	0.7983	0.852


Scores for ddi with type int
tp	fp	fn	total	prec	recall	F1
2	1	0	2	0.6667	1	0.8


MACRO-AVERAGE MEASURES FOR DDI CLASSIFICATION:
	P	R	F1
	0.8548	0.8417	0.8482
________________________________

In [17]:
interaction_classifier(TRAIN_DIR, TRAIN_DIR, OUTPUT_FILE)


Gold project/data/Train
Submission  task9.2_VJ_42.txt

project/data/Train
 Gold DDI standard saved in goldDDI.txt total pairs: 25089
SCORES FOR THE GROUP: VJ RUN=42
Gold Dataset: /Train

Partial Evaluation: only detection of DDI (regadless to the type)
tp	fp	fn	total	prec	recall	F1
3514	0	22	3536	1	0.9938	0.9969


Detection and classification of DDI
tp	fp	fn	total	prec	recall	F1
3497	17	39	3536	0.9952	0.989	0.9921


________________________________________________________________________

SCORES FOR DDI TYPE
Scores for ddi with type mechanism
tp	fp	fn	total	prec	recall	F1
1109	2	9	1118	0.9982	0.9919	0.9951


Scores for ddi with type effect
tp	fp	fn	total	prec	recall	F1
1500	0	25	1525	1	0.9836	0.9917


Scores for ddi with type advise
tp	fp	fn	total	prec	recall	F1
704	5	3	707	0.9929	0.9958	0.9944


Scores for ddi with type int
tp	fp	fn	total	prec	recall	F1
184	10	2	186	0.9485	0.9892	0.9684


MACRO-AVERAGE MEASURES FOR DDI CLASSIFICATION:
	P	R	F1
	0.9849	0.9901	0.9875
___________________

In [18]:
interaction_classifier(TRAIN_DIR, TEST_DIR, OUTPUT_FILE)


Gold project/data/Test-DDI
Submission  task9.2_VJ_42.txt

project/data/Test-DDI
 Gold DDI standard saved in goldDDI.txt total pairs: 5716
SCORES FOR THE GROUP: VJ RUN=42
Gold Dataset: /Test-DDI

Partial Evaluation: only detection of DDI (regadless to the type)
tp	fp	fn	total	prec	recall	F1
853	0	126	979	1	0.8713	0.9312


Detection and classification of DDI
tp	fp	fn	total	prec	recall	F1
726	127	253	979	0.8511	0.7416	0.7926


________________________________________________________________________

SCORES FOR DDI TYPE
Scores for ddi with type mechanism
tp	fp	fn	total	prec	recall	F1
231	36	71	302	0.8652	0.7649	0.812


Scores for ddi with type effect
tp	fp	fn	total	prec	recall	F1
266	60	94	360	0.816	0.7389	0.7755


Scores for ddi with type advise
tp	fp	fn	total	prec	recall	F1
191	22	30	221	0.8967	0.8643	0.8802


Scores for ddi with type int
tp	fp	fn	total	prec	recall	F1
38	9	58	96	0.8085	0.3958	0.5315


MACRO-AVERAGE MEASURES FOR DDI CLASSIFICATION:
	P	R	F1
	0.8466	0.691	0.7609
__________